# Inkriff — Extracción completa de datos

Versión escalada del notebook piloto (`inkriff_pilot_extraccion.ipynb`), que ya
validamos: usa el MISMO codigo probado (MusicBrainz con reintentos, Genius solo
metadatos oficiales, Open Library con Google Books de respaldo), pero corre sobre
**todo el universo semilla** (~81 bandas, ~112 libros) en vez de solo una
muestra.

Esta version (v2) ya incluye las bandas y libros de `pares_semilla_final.csv` — los 105
pares libro<->banda que curamos manualmente a partir de nuestro propio conocimiento de fandom — además
del universo original. Asi la extraccióm cubre también los anclajes que vamos a usar
como entrenamiento supervisado del dual encoder, no solo la muestra inicial.

Cada banda y cada libro queda etiquetado con su subgénero/categoría semilla, para que
el CSV resultante ya se pueda usar directo en la siguiente fase (EDA / ingenieria de
variables).

**Tiempo estimado:** unos 10-15 minutos en total (MusicBrainz es la fuente mas lenta por
su limite de tasa — son ~81 bandas x ~3s cada una, mas reintentos; es normal ver
algún `[MusicBrainz 503] reintentando...` en el camino. Si Colab se desconecta a mitad,
solo vuelve a correr las celdas en orden).

Corre las celdas en orden.


In [ ]:
!pip -q install requests


## 0. Universo de datos semilla completo


In [ ]:
SEED_BANDS = {'Hard Rock / Rock clasico': ['Led Zeppelin', 'AC/DC', 'Deep Purple'],
 'Heavy Metal': ['Iron Maiden', 'Judas Priest', 'Black Sabbath', 'Avenged Sevenfold'],
 'Power Metal': ['Blind Guardian', 'DragonForce', 'Sabaton', 'Rhapsody of Fire'],
 'Symphonic Metal': ['Nightwish', 'Epica', 'Within Temptation', 'Kamelot'],
 'Folk Metal': ['Eluveitie', 'Korpiklaani', 'Wintersun'],
 'Gothic Metal': ['Type O Negative',
                  'Paradise Lost',
                  'Lacuna Coil',
                  'Evanescence',
                  'Chelsea Wolfe'],
 'Progressive Metal': ['Dream Theater', 'Opeth', 'Tool', 'Gojira'],
 'Death Metal': ['Death', 'Cannibal Corpse', 'At the Gates'],
 'Black Metal': ['Mayhem', 'Emperor', 'Dimmu Borgir'],
 'Metalcore': ['Killswitch Engage', 'Bring Me The Horizon', 'Architects', 'Spiritbox', 'Jinjer'],
 'Deathcore': ['Suicide Silence', 'Whitechapel', 'Lorna Shore'],
 'Nu Metal': ['Slipknot', 'Korn', 'System of a Down'],
 'Doom Metal / Stoner': ['Electric Wizard', 'Sleep', 'Candlemass'],
 'Post-Metal / Atmospheric': ['Alcest', 'Agalloch'],
 'Melodic Death Metal': ['Amon Amarth'],
 'Blackened Death Metal': ['Behemoth'],
 'Post-Hardcore': ['Pierce the Veil', 'Sleeping With Sirens'],
 'Alternative Metal': ['Sleep Token',
                       'Deftones',
                       'Bad Omens',
                       'Breaking Benjamin',
                       'Disturbed',
                       'Linkin Park'],
 'Emo / Alternative Rock': ['My Chemical Romance'],
 'Pop Punk': ['Fall Out Boy',
              'The Wonder Years',
              'Sum 41',
              'All Time Low',
              'Paramore',
              'Blink-182',
              'Neck Deep',
              'Simple Plan',
              'Mayday Parade'],
 'Alternative Rock': ['The Maine',
                      'The 1975',
                      'Florence + The Machine',
                      'The Smashing Pumpkins',
                      'Muse',
                      'The Pretty Reckless',
                      'Royal Blood'],
 'Gothic Rock': ['The Cure', 'HIM', 'The 69 Eyes'],
 'Occult Rock': ['Ghost'],
 'Punk Rock': ['Green Day']}

SEED_BOOKS = {'Alta fantasia / epica': ['El Senor de los Anillos',
                           'Canción de Hielo y Fuego',
                           'El Nombre del Viento',
                           'El Hobbit',
                           'El Silmarillion',
                           'Eragon',
                           'El temor de un hombre sabio',
                           'Rey de cicatrices',
                           'Mistborn: El imperio final',
                           'El pozo de la ascensión',
                           'El héroe de las eras',
                           'El camino de los reyes',
                           'El archivo de las tormentas',
                           'La rueda del tiempo',
                           'El priorato del naranjo'],
 'Romantasy (fantasia romantica)': ['Trono de Cristal',
                                    'Una Corte de Rosas y Espinas',
                                    'Cuarto Ala',
                                    'Una corte de niebla y furia',
                                    'Una corte de alas y ruina',
                                    'Reina de sombras',
                                    'Fourth Wing',
                                    'Iron Flame',
                                    'Onyx Storm',
                                    'Powerless',
                                    'Reckless',
                                    'Divine Rivals',
                                    'Ruthless Vows',
                                    'Érase una vez un corazón roto',
                                    'La balada del príncipe roto',
                                    'La vida invisible de Addie LaRue',
                                    'Stardust',
                                    'Carry On'],
 'Fantasia oscura / gotica': ['Sombra y Hueso',
                              'Drácula',
                              'Vicious',
                              'Berserk',
                              'The Witcher',
                              'La novena casa',
                              'Hell Bent',
                              'El océano al final del camino',
                              'Elric de Melniboné',
                              'The Poppy War',
                              'La guerra de la amapola',
                              'Coraline',
                              'A Darker Shade of Magic',
                              'The Locked Tomb',
                              'Gideon the Ninth',
                              'Mexican Gothic',
                              'The Hazel Wood',
                              'Jonathan Strange y el señor Norrell',
                              'Piranesi'],
 'Romance historico / gotico': ['Outlander', 'Jane Eyre', 'Un día de diciembre'],
 'Fantasia urbana': ['Cazadores de Sombras: Ciudad de Hueso',
                     'The Infernal Devices',
                     'Cazadores de sombras',
                     'Ciudad de hueso',
                     'Seis de cuervos'],
 'Mitologia (nordica / celta / griega)': ['Mitología Nórdica',
                                          'La canción de Aquiles',
                                          'Circe',
                                          'Percy Jackson',
                                          'El ladrón del rayo',
                                          'Héroes del Olimpo',
                                          'Gods of Jade and Shadow'],
 'Fae Fantasy': ['El príncipe cruel', 'El rey malvado', 'La reina de nada'],
 'Fantasia clasica / juvenil': ['Las crónicas de Narnia',
                                'La brújula dorada',
                                'El castillo ambulante',
                                'Alicia en el País de las Maravillas',
                                'A través del espejo',
                                'Caraval',
                                'El circo de la noche',
                                'El mar sin estrellas'],
 'Grimdark': ['La primera ley', 'Malaz: El libro de los caídos', 'La compañía negra'],
 'Dark Academia': ['Babel', 'Ninth House'],
 'Gotico clasico / vampiros': ['Entrevista con el vampiro',
                               'El vampiro Lestat',
                               'Las brujas de Mayfair',
                               'Carmilla',
                               'El retrato de Dorian Gray',
                               'Frankenstein',
                               'El castillo de Otranto',
                               'Vampire Academy',
                               'A Dowry of Blood'],
 'Distopia juvenil': ['Los juegos del hambre',
                      'En llamas',
                      'Sinsajo',
                      'Divergente',
                      'Maze Runner',
                      'Ready Player One',
                      'Scott Pilgrim'],
 'Romance juvenil / coming-of-age': ['Las ventajas de ser invisible',
                                     'Heartstopper',
                                     'A todos los chicos de los que me enamoré',
                                     'Eleanor & Park'],
 'Fantasia de cuento de hadas': ['Nettle & Bone',
                                 'Uprooted',
                                 'Spinning Silver',
                                 'A Deadly Education',
                                 'The Bear and the Nightingale',
                                 'The Once and Future Witches',
                                 'The Invisible Life of Addie LaRue'],
 'Cozy Fantasy / sobrenatural': ['La casa en el mar más azul', 'Under the Whispering Door']}

# Diccionarios inversos: banda/libro -> su subgenero/categoria semilla
BAND_TO_SUBGENRE = {banda: sg for sg, bandas in SEED_BANDS.items() for banda in bandas}
BOOK_TO_CATEGORY = {libro: cat for cat, libros in SEED_BOOKS.items() for libro in libros}

ALL_BANDS = list(BAND_TO_SUBGENRE.keys())
ALL_BOOKS = list(BOOK_TO_CATEGORY.keys())
print(f"{len(ALL_BANDS)} bandas | {len(ALL_BOOKS)} libros")


## 1. MusicBrainz — todas las bandas

Requiere un `User-Agent` personalizado y respetar el limite de ~1 request/segundo
(uso anonimo). No requiere API key. Reintenta automaticamente ante 503/429.


In [ ]:
import time
import requests

MB_HEADERS = {"User-Agent": "InkriffDataCollector/0.1 (contacto: tu_correo@ejemplo.com)"}
MB_BASE = "https://musicbrainz.org/ws/2"


def mb_get(url, params, max_retries=5):
    """GET a un endpoint de MusicBrainz con reintento + backoff ante 503/429."""
    for intento in range(max_retries):
        r = requests.get(url, params=params, headers=MB_HEADERS, timeout=20)
        if r.status_code in (503, 429):
            espera = 2 ** intento  # 1, 2, 4, 8, 16 segundos
            print(f"  [MusicBrainz {r.status_code}] reintentando en {espera}s...")
            time.sleep(espera)
            continue
        r.raise_for_status()
        return r.json()
    raise RuntimeError(f"MusicBrainz no respondio tras {max_retries} intentos: {url}")


def mb_search_artist(name):
    data = mb_get(f"{MB_BASE}/artist", {"query": f'artist:"{name}"', "fmt": "json", "limit": 1})
    return data["artists"][0] if data.get("artists") else None


def mb_artist_detail(mbid):
    return mb_get(f"{MB_BASE}/artist/{mbid}", {"fmt": "json", "inc": "tags+url-rels"})


mb_results = []
for name in ALL_BANDS:
    try:
        artist = mb_search_artist(name)
        time.sleep(1.5)  # un poco mas holgado que el minimo de 1 req/seg
        if not artist:
            print(f"[!] No encontrado en MusicBrainz: {name}")
            continue
        detail = mb_artist_detail(artist["id"])
        time.sleep(1.5)
        tags = sorted(detail.get("tags", []), key=lambda t: t.get("count", 0), reverse=True)
        spotify_url = next(
            (rel["url"]["resource"] for rel in detail.get("relations", [])
             if "spotify.com" in rel.get("url", {}).get("resource", "")),
            None,
        )
        row = {
            "banda": name,
            "subgenero_semilla": BAND_TO_SUBGENRE[name],
            "mbid": artist["id"],
            "pais": detail.get("country"),
            "tags_top": [t["name"] for t in tags[:5]],
            "spotify_url": spotify_url,
        }
        mb_results.append(row)
        print(row)
    except Exception as e:
        print(f"[!] Error con {name}: {e}")
        continue

print(f"\nTotal de bandas con datos de MusicBrainz: {len(mb_results)} de {len(ALL_BANDS)}")


## 2. Genius — todas las bandas (API oficial, solo metadatos)

Igual que en el piloto: solo metadatos (titulo, popularidad), nunca letras — Genius
bloquea el scraping con Cloudflare/reCAPTCHA (403), ademas de que iba contra sus
Terminos de Servicio.


In [ ]:
GENIUS_TOKEN = "PEGA_AQUI_TU_TOKEN"  # https://genius.com/api-clients
GENIUS_HEADERS = {"Authorization": f"Bearer {GENIUS_TOKEN}"}


def genius_search_song(name):
    r = requests.get(
        "https://api.genius.com/search",
        params={"q": name},
        headers=GENIUS_HEADERS, timeout=15,
    )
    r.raise_for_status()
    hits = r.json().get("response", {}).get("hits", [])
    return hits[0]["result"] if hits else None


genius_results = []
for name in ALL_BANDS:
    try:
        result = genius_search_song(name)
        time.sleep(0.5)
        if not result:
            print(f"[!] No encontrado en Genius: {name}")
            continue
        row = {
            "banda_buscada": name,
            "titulo_top": result.get("title"),
            "artista_genius": result.get("primary_artist", {}).get("name"),
            "genius_id": result.get("id"),
            "pageviews": result.get("stats", {}).get("pageviews"),
        }
        genius_results.append(row)
        print(row)
    except Exception as e:
        print(f"[!] Error con {name}: {type(e).__name__}: {e}")

print(f"\nTotal de bandas con datos de Genius: {len(genius_results)} de {len(ALL_BANDS)}")


## 3. Open Library — todos los libros (con Google Books como respaldo)

**Nota:** igual que con MusicBrainz, es normal que Open Library o Google Books tarden
o no respondan de vez en cuando (`ConnectTimeout`, `ReadTimeout`) — son APIs publicas
compartidas por muchisima gente, sobre todo desde Colab. El codigo de abajo reintenta
automaticamente con backoff, y si una fuente sigue sin responder despues de varios
intentos, lo dice mas no truena el resto del ciclo.


In [ ]:
def get_con_reintento(url, params, max_retries=4, timeout=20):
    """GET con reintento + backoff ante 503/429 y ante fallas de conexion/timeout."""
    for intento in range(max_retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            if r.status_code in (503, 429):
                espera = 2 ** intento
                print(f"  [{r.status_code}] reintentando en {espera}s...")
                time.sleep(espera)
                continue
            r.raise_for_status()
            return r.json()
        except (requests.exceptions.ConnectTimeout, requests.exceptions.ReadTimeout,
                requests.exceptions.ConnectionError) as e:
            espera = 2 ** intento
            print(f"  [{type(e).__name__}] reintentando en {espera}s...")
            time.sleep(espera)
    raise RuntimeError(f"No se pudo conectar tras {max_retries} intentos: {url}")


def openlibrary_search(title):
    try:
        data = get_con_reintento("https://openlibrary.org/search.json", {"q": title, "limit": 1})
        docs = data.get("docs", [])
        return docs[0] if docs else None
    except Exception as e:
        print(f"  [Open Library] no disponible para \"{title}\": {type(e).__name__}: {e}")
        return None


def googlebooks_search(title):
    try:
        data = get_con_reintento("https://www.googleapis.com/books/v1/volumes", {"q": title, "maxResults": 1})
        items = data.get("items", [])
        return items[0] if items else None
    except Exception as e:
        print(f"  [Google Books] no disponible para \"{title}\": {type(e).__name__}: {e}")
        return None


book_results = []
for title in ALL_BOOKS:
    ol = openlibrary_search(title)
    row = {"titulo_buscado": title, "categoria_semilla": BOOK_TO_CATEGORY[title]}
    if ol:
        row.update({
            "titulo": ol.get("title"),
            "autor": ", ".join(ol.get("author_name", [])),
            "anio": ol.get("first_publish_year"),
            "fuente": "Open Library",
        })
    else:
        gb = googlebooks_search(title)
        if gb:
            info = gb.get("volumeInfo", {})
            row.update({
                "titulo": info.get("title"),
                "autor": ", ".join(info.get("authors", [])),
                "anio": info.get("publishedDate"),
                "fuente": "Google Books",
            })
        else:
            row.update({"titulo": None, "autor": None, "anio": None, "fuente": None})
    book_results.append(row)
    print(row)
    time.sleep(0.3)

print(f"\nTotal de libros con datos: {len(book_results)} de {len(ALL_BOOKS)}")


## 4. Consolidar dataset completo

Guarda 3 CSVs por fuente, mas un cuarto (`bandas_completo.csv`) que ya junta
MusicBrainz + Genius por banda, listo para la siguiente fase.


In [ ]:
import pandas as pd

df_bandas = pd.DataFrame(mb_results)
df_genius = pd.DataFrame(genius_results)
df_libros = pd.DataFrame(book_results)

df_bandas.to_csv("bandas_musicbrainz.csv", index=False)
df_genius.to_csv("metadatos_genius.csv", index=False)
df_libros.to_csv("libros.csv", index=False)

df_bandas_completo = df_bandas.merge(
    df_genius, left_on="banda", right_on="banda_buscada", how="left"
).drop(columns=["banda_buscada"])
df_bandas_completo.to_csv("bandas_completo.csv", index=False)

print("Bandas (MusicBrainz + Genius):")
display(df_bandas_completo)
print("Libros:")
display(df_libros)
